<a href="https://colab.research.google.com/github/amilynestes1028/ds2002-fa26/blob/main/notebooks/04-jsons-apis/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** One row represents an order line item: a soecific item and its quantity within a given order, combined with the vendor details

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)
print(lines)

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id']) flattens the nested lines array, creating a single row for each individual line item while duplicating the parent order and vendor_id fields alongside it. This produces a DataFrame with 4 rows across the three orders and a combined total qty of 7, satisfying both assertions.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
joined = lines.merge(
    pd.json_normalize(vendors_json),
    on='vendor_id',
    how='left',
    indicator=True
)

print(joined)
# Report the vendor that did not match and how many units are attached to it
unmatched = joined[joined['_merge'] == 'left_only']
print("Unmatched vendor:", unmatched['vendor_id'].iloc[0])
print("Units attached:", unmatched['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
Unmatched vendor: V-07
Units attached: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
totals = (
    joined
    .groupby('name', dropna=False)['qty']
    .sum()
    .reset_index()
)

# The unmatched vendor (V-07) appears as NaN because it was not in vendors_json.
print(totals)

           name  qty
0     Cav Merch    1
1  Hoos Burgers    3
2           NaN    3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
zone_totals = (
    joined
    .groupby('zone', dropna=False)['qty']
    .sum()
    .reset_index()
)

print(zone_totals)

  zone  qty
0    A    4
1  NaN    3


The total across all named zones is 4 items (Zone A). The remaining 3 items were sold under Order 3 by vendor V-07, which is missing from vendors_json and therefore has its zone set to NaN. Because standard pandas grouping functions ignore missing values by default when aggregating across a column like zone, those 3 units are left out of the individual zone sums, accounting for the difference between the named zone total of 4 and the dataset's total quantity of 7.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged_with_lines = [x for x in ragged_json if 'lines' in x]

lines2 = pd.json_normalize(
    ragged_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(lines2)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is useful because each individual item and its quantity can be analyzed separately while still keeping the order and vendor information attached. This makes asking like “How many total units of each item were sold?” easy to answer, which would be harder if multiple items were stored together in one row per order.

b)A missing quantity means the quantity is unknown or was not provided, while a quantity of zero means the item was recorded but none were sold. If they are treated as the same, downstream totals and averages can be wrong because missing data could be incorrectly counted as zero sales, which can distort vendor, item, and overall sales calculations.